# H&M Transaction Data: Product Recommendations 01
## Pre-process data

**Purpose**  
This notebook engineers features and constructs the labeled datasets used to train the purchase prediction model. Raw data has already been cleaned in a prior step. The output — train, validation, and test sets are saved as `.pkl` files and loaded in the next notebook.

**Temporal Structure**  
Each dataset is built around two non-overlapping time windows:
- **History window** (30 days): used to compute customer and product features
- **Prediction window** (14 days): defines the labels — did a customer purchase a product during this period?

All features are computed strictly from data before the prediction window to prevent leakage.

**Labels**  
- Positive examples (`purchased = 1`): all unique customer-product pairs observed in the prediction window  
- Negative examples (`purchased = 0`): randomly sampled customer-product pairs with no purchase in the prediction window. Because the space of possible pairs is enormous, we sample at a fixed ratio rather than exhaustively enumerating all non-purchases.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

# Define paths
base_path = Path("../data/")
raw_path = processed_path = base_path / 'raw'
processed_path = base_path / 'processed'

# Feature Enginering Classes

These feature engineering classes are saved into `/src` with test, but copied here for clarity.

- `CustomerFeatureEngineer` — computes RFM, behavioral, and demographic features per customer
- `ProductFeatureEngineer` — computes popularity and price features per product
- `RecommendationTrainingBuilder` — assembles positive and negative training examples by joining customer and product features

In [2]:
import pandas as pd
import numpy as np

class CustomerFeatureEngineer:
    def __init__(self, customers_df, transactions_df):
        self.customers = customers_df
        self.transactions = transactions_df
        self.set_dtypes()
    
    def calculate_rfm_and_behaviors(self, as_of_date = None):
        """
        days_since_last_purchase
        num_purchases
        total_spent
        avg_transaction_value
        customer_price_std
        avg_days_between purchases
        """
        as_of_date = self._as_of_date(as_of_date)

        relevant_txn = self._relevant_transactions(as_of_date)
        agg_data = relevant_txn.groupby('customer_id').agg(
            first_purchase_date = ('t_dat', 'min'),
            last_purchase_date = ('t_dat', 'max'),
            customer_price_std = ('price', 'std'),
            num_purchases = ('t_dat', 'count'),
            total_spent = ('price','sum')
        ).reset_index()

        customer_cols = ['customer_id']
        if 'age' in self.customers.columns:
            customer_cols.append('age')

        agg_data = self.customers[customer_cols].merge(
            agg_data,
            on='customer_id',
            how='left'
        )
        agg_data['num_purchases'] = agg_data['num_purchases'].fillna(0).astype(int)
        agg_data['total_spent'] = agg_data['total_spent'].fillna(0)
       
        # Low numbers are good, so we fill with arbitrary high value
        agg_data['days_since_last_purchase'] = (as_of_date - agg_data['last_purchase_date']).dt.days
        agg_data['days_since_last_purchase'] = agg_data['days_since_last_purchase'].fillna(999) 
        
        agg_data['avg_transaction_value'] = (agg_data['total_spent'] / agg_data['num_purchases']).fillna(0)
        
        # days between purchases is also better at lower so fill with max + 1
        agg_data['avg_days_between_purchases'] = (agg_data['last_purchase_date'] - agg_data['first_purchase_date']).dt.days / (agg_data['num_purchases']-1)
        # fillna with the max doesn't work when for one purchase only 0 / 0
        agg_data['avg_days_between_purchases'] = agg_data['avg_days_between_purchases'].fillna(999)

        agg_data['customer_price_std'] = agg_data['customer_price_std'].fillna(0)

        agg_data = agg_data.drop('last_purchase_date', axis=1)
        agg_data = agg_data.drop('first_purchase_date', axis=1)

        return agg_data


    def calculate_category_preferences(self, articles_df, as_of_date=None):
        """
        favorite_department
        favorite_garment_group
        category_diversity
        """

        def category_mode(category):
            return category.mode()[0] if not category.mode().empty else None

        as_of_date = self._as_of_date(as_of_date)
        
        relevant_txn = self._relevant_transactions(as_of_date)
        txn_with_category = relevant_txn.merge(
            articles_df[['article_id','department_name', 'garment_group_name']],
            on = 'article_id',
            how = 'left'
        )
        category_df = txn_with_category.groupby('customer_id').agg(
            primary_department = ('department_name', category_mode),
            primary_garment_group = ('garment_group_name', category_mode),
            category_diversity = ('department_name', 'nunique')
        ).reset_index()

        category_df = self.customers[['customer_id']].merge(
            category_df,
            on='customer_id',
            how='left'
        )
        category_df['primary_department'] = category_df['primary_department'].fillna('Unknown')
        category_df['primary_garment_group'] = category_df['primary_garment_group'].fillna('Unknown')
        category_df['category_diversity'] = category_df['category_diversity'].fillna(0).astype(int)

        return category_df
   
    def calculate_cold_start_features(self, as_of_date=None):
        as_of_date = self._as_of_date(as_of_date)
        relevant_txn = self._relevant_transactions(as_of_date)
        cold_start_df = self.customers.copy()
        
        cold_start_df['is_new_customer'] = (~cold_start_df['customer_id'].isin(relevant_txn['customer_id'])).astype(int)

        if 'club_member_status' in cold_start_df.columns:
            cold_start_df['club_member_status'] = cold_start_df['club_member_status'].replace('LEFT CLUB', 'NOT_ACTIVE_MEMBER')
            club_member_dummy = pd.get_dummies(cold_start_df['club_member_status'], prefix='club_member_status', dtype=int, drop_first=False)
            if 'club_member_status_ACTIVE' in club_member_dummy.columns:
                club_member_dummy = club_member_dummy.drop('club_member_status_ACTIVE', axis=1)
            cold_start_df = pd.concat([cold_start_df, club_member_dummy ], axis=1)
       
        if 'fashion_news_frequency' in cold_start_df.columns:
            cold_start_df['fashion_news_frequency'] = cold_start_df['fashion_news_frequency'].replace("MONTHLY", "REGULARLY")
            fashion_news_dummy = pd.get_dummies(cold_start_df['fashion_news_frequency'], prefix='fashion_news_frequency', dtype=int, drop_first=False)
            if 'fashion_news_frequency_NONE' in fashion_news_dummy.columns:
                fashion_news_dummy = fashion_news_dummy.drop('fashion_news_frequency_NONE', axis=1)
            cold_start_df = pd.concat([cold_start_df, fashion_news_dummy], axis=1)
            
            cold_start_df = cold_start_df.drop(['fashion_news_frequency', 'club_member_status'], axis=1)

        cols_to_keep = ['customer_id', 'is_new_customer']
        optional_cols = ['FN', 'Active', 'club_member_status_NOT_ACTIVE_MEMBER', 
                         'club_member_status_PRE-CREATE', 'fashion_news_frequency_REGULARLY']
        cols_to_keep += [col for col in optional_cols if col in cold_start_df.columns]
        return cold_start_df[cols_to_keep]

    def get_fill_values(self):
        return {
            'num_purchases': 0,
            'total_spent': 0,
            'avg_transaction_value': 0,
            'customer_price_std': 0,
            'days_since_last_purchase': 999,
            'avg_days_between_purchases': 999,
            'category_diversity': 0,
            'primary_department': 'Unknown',
            'primary_garment_group': 'Unknown',
            'FN': 0,
            'Active': 0,
            'is_new_customer': 1,
            'club_member_status_NOT_ACTIVE_MEMBER': 0,
            'club_member_status_PRE-CREATE': 0,
            'age': 0,
            'fashion_news_frequency_REGULARLY': 0
        }

    def _as_of_date(self, as_of_date=None):
        if as_of_date == None:
            as_of_date = self.transactions['t_dat'].max()
        return pd.to_datetime(as_of_date)

    def _relevant_transactions(self, as_of_date):
        return self.transactions[self.transactions['t_dat'] <= as_of_date]

    def calculate_all_features(self, articles_df, as_of_date=None):
        rfm_behaviors = self.calculate_rfm_and_behaviors(as_of_date)
        categories = self.calculate_category_preferences(articles_df, as_of_date)
        cold_start = self.calculate_cold_start_features(as_of_date)

        all_features = rfm_behaviors.merge(
            categories,
            on='customer_id',
            how = 'left'
        ).reset_index(drop=True)

        all_features = all_features.merge(
            cold_start,
            on='customer_id',
            how = 'left'
        ).reset_index(drop=True)

        return all_features

    def set_dtypes(self):
        self.transactions['t_dat'] = pd.to_datetime(self.transactions['t_dat'])

In [3]:
import pandas as pd
import numpy as np

class ProductFeatureEngineer:
    def __init__(self, articles_df, transactions_df):
        self.articles = articles_df
        self.transactions = transactions_df
        self.set_dtypes()
    
    def set_dtypes(self):
        self.transactions['t_dat'] = pd.to_datetime(self.transactions['t_dat'])
    
    def calculate_popularity(self, as_of_date=None):
        """
        - sales_last_7_days
        - sales_last_30_days
        - days_since_first_sale
        """
        if as_of_date==None:
            as_of_date = self.transactions['t_dat'].max()
        as_of_date = pd.to_datetime(as_of_date)

        relevant_txn = self.transactions[self.transactions['t_dat'] <= as_of_date]
        
        last_7_days_txn = relevant_txn[relevant_txn['t_dat'] >= (as_of_date - pd.Timedelta(days=7))]
        last_30_days_txn = relevant_txn[relevant_txn['t_dat'] >= (as_of_date - pd.Timedelta(days=30))]

        sales_7d = last_7_days_txn.groupby('article_id').size().reset_index(name='sales_last_7_days')
        sales_30d = last_30_days_txn.groupby('article_id').size().reset_index(name='sales_last_30_days')
        sale_dates = relevant_txn.groupby('article_id').agg(
            first_sale_date=('t_dat', 'min'),
            last_sale_date=('t_dat', 'max')
        ).reset_index()
        
        popularity = sale_dates.merge(sales_7d, on='article_id', how='left')
        popularity = popularity.merge(sales_30d, on='article_id', how='left')
        
        popularity['sales_last_7_days'] = popularity['sales_last_7_days'].fillna(0).astype(int)
        popularity['sales_last_30_days'] = popularity['sales_last_30_days'].fillna(0).astype(int)

        # fillna with 0, treat it like a brand new product with no history
        popularity['days_since_first_sale'] = (as_of_date - popularity['first_sale_date']).dt.days.fillna(0).astype(int)
        
        # fillna with max, treat it like an inactive product
        popularity['days_since_last_sale'] = (as_of_date - popularity['last_sale_date']).dt.days.astype(int)
        max_last_sale = popularity['days_since_last_sale'].max() + 1
        popularity['days_since_last_sale'] = popularity['days_since_last_sale'].fillna(max_last_sale)

        popularity = popularity.drop(['first_sale_date', 'last_sale_date'], axis=1)

        return popularity
    
    def calculate_price_features(self, as_of_date=None):
        """
        - avg_price
        - product_price_std
        - min_price
        - max_price
        """
        if as_of_date==None:
            as_of_date = self.transactions['t_dat'].max()
        as_of_date = pd.to_datetime(as_of_date)

        relevant_txn = self.transactions[self.transactions['t_dat'] <= as_of_date]
        
        agg_data = relevant_txn.groupby('article_id').agg(
            avg_price = ('price', 'mean'),
            min_price = ('price', 'min'),
            max_price = ('price', 'max'),
            product_price_std = ('price', 'std'),
        ).reset_index()


        global_avg_price = relevant_txn['price'].mean()
        agg_data['avg_price'] = agg_data['avg_price'].fillna(global_avg_price)
        agg_data['min_price'] = agg_data['min_price'].fillna(global_avg_price)
        agg_data['max_price'] = agg_data['max_price'].fillna(global_avg_price)
        agg_data['product_price_std'] = agg_data['product_price_std'].fillna(0)

        return agg_data

    def get_fill_values(self):
        return {
            'sales_last_7_days': 0,
            'sales_last_30_days': 0,
            'days_since_first_sale': 999,
            'days_since_last_sale': 999,
            'avg_price': 0,
            'min_price': 0,
            'max_price': 0,
            'product_price_std': 0
        }

    def calculate_all_features(self, as_of_date=None):
        """Combine popularity and price features"""
        popularity = self.calculate_popularity(as_of_date=as_of_date)
        price_features = self.calculate_price_features(as_of_date=as_of_date)
        
        all_features = popularity.merge(price_features, on='article_id', how='left')
        
        return all_features

In [4]:
import pandas as pd
import numpy as np

class RecommendationTrainingBuilder:
    def __init__(self, transactions_df, customer_features_df, product_features_df, customer_fill_values={}, product_fill_values={}):
        self.transactions = transactions_df
        self.customer_features = customer_features_df
        self.product_features = product_features_df
        self.customer_fill_values = customer_fill_values
        self.product_fill_values = product_fill_values

    def build_dataset(self, prediction_start=None, prediction_end=None, negative_ratio=2, random_state=67):
        """
        Creates a data set of customer-product pairs
        
        Outcome variable:
        purchased
        - 1 = purchased during the prediction period
        - 0 = did NOT purchase during the prediction period

        Method: build_dataset
        builds product and customer features
        prediction_start = Begin prediction period
        prediction_end = End of prediction period
        negative_ratio = Ratio of random pairs of customer-product that did not result in a transaction

        Returns:
            pd.DataFrame of features and outcome
        """
        positives = self._get_positive_examples(prediction_start, prediction_end)
        negatives = self._sample_negative_examples(positives, negative_ratio=negative_ratio, random_state=random_state)
        return pd.concat([positives, negatives], axis=0)
    
    def _get_positive_examples(self, prediction_start=None, prediction_end=None):
        if prediction_start == None:
            prediction_start = self.transactions['t_dat'].min()
        prediction_start = pd.to_datetime(prediction_start)
        if prediction_end == None:
            prediction_end = self.transactions['t_dat'].max()
        prediction_end = pd.to_datetime(prediction_end)
        start_mask = self.transactions['t_dat']<= prediction_end
        end_mask = self.transactions['t_dat']>= prediction_start
        relevant_txn = self.transactions[start_mask & end_mask]
        
        positives = relevant_txn[['customer_id', 'article_id']].drop_duplicates()
        positives['purchased'] = 1
        positives = positives.merge(
            self.product_features,
            on='article_id',
            how='left'
        )
        positives = positives.merge(
            self.customer_features,
            on='customer_id',
            how='left'
        )
        if 'num_purchases' in positives.columns:
            positives['is_new_customer'] = positives['num_purchases'].isnull().astype(int)
        else:
            positives['is_new_customer'] = 0
        
        positives = self.fillna(positives)
        return positives
    
    def _sample_negative_examples(self, positives, negative_ratio=2, random_state=67):
        rng = np.random.default_rng(random_state)

        article_ids = self.product_features['article_id'].unique()
        customer_ids = positives['customer_id'].unique()

        positives_per_customer = positives.groupby('customer_id').size()

        negative_examples = []
        for customer in customer_ids:
            num_positives = positives_per_customer[customer]
            num_negatives = num_positives * negative_ratio

            bought = positives[positives['customer_id'] == customer]['article_id'].values
            buffer_size = min(num_negatives*2, len(article_ids))
            sampled_articles = rng.choice(article_ids, size=buffer_size, replace=False)
            sampled_articles = sampled_articles[~np.isin(sampled_articles, list(bought))][:num_negatives]
            
            negative_examples.extend(
                {
                    'customer_id': customer,
                    'article_id': product,
                    'purchased': 0
                }
                for product in sampled_articles
            )

        negatives = pd.DataFrame(negative_examples)
        negatives = negatives.merge(
            self.product_features,
            on='article_id',
            how='left'
        )
        negatives = negatives.merge(
            self.customer_features,
            on='customer_id',
            how='left'
        )
        negatives = self.fillna(negatives)

        return negatives

    def fillna(self, df):
        if self.customer_fill_values:
            df = df.fillna(self.customer_fill_values)
        if self.product_fill_values:
            df = df.fillna(self.product_fill_values)
        return df


## Load Cleaned Data

Loads the three cleaned datasets produced by the EDA/cleaning notebook (`0_eda_data_cleaning`):
- `articles_hm_cleaned.csv` — 105,542 products with metadata (department, garment group, etc.)
- `customers_hm_cleaned.csv` — 1,048,575 customers with demographics and membership info
- `transactions_hm_cleaned.csv` — 1,040,101 purchase records with date and price

Note: we work with a sample of the full H&M dataset.

In [5]:
customers = pd.read_csv(processed_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(processed_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(processed_path / 'articles_hm_cleaned.csv')

print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

# convert transaction date to datetime
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


## Train, Validation Data Generation

### Time-Based Train/Validation/Test Split
Rather than a random split, we use a **temporal split** to simulate real-world deployment of training on historical data and predicting future events.

Each dataset is built over a distinct, non-overlapping time windows. Train data comes from 3 separate and non-overlapping time periods:

| Split      | History Window        | Prediction Window       | Negative Ratio |
|------------|-----------------------|-------------------------|----------------|
| Train 1    | Jan 2019              | Feb 1–14, 2019          | 1:1            |
| Train 2    | Mar 2019              | Apr 1–14, 2019          | 1:1            |
| Train 3    | May 2019              | Jun 1–14, 2019          | 1:1            |
| Validation | Jul 2019              | Aug 1–14, 2019          | 5:1            |
| Test       | Sep 2019              | Oct 1–14, 2019          | 5:1            |

Using multiple time periods increases the diversity of training examples and reduces sensitivity to seasonal effects in any single window. The three training datasets are concatenated into a single training set after construction.

Training uses 1:1 to ensure the model has a clear view of positive examples without being overwhelmed by negative examples. Validation and test use 5:1 to better reflect the real-world class imbalance and give a more realistic picture of model performance.

In [6]:
HISTORY_DAYS = 30

def build_product_recommendation_data(prediction_start_date, prediction_end_date, history_days=90, negative_ratio=5,random_state=67):
    as_of_date = pd.to_datetime(prediction_start_date) - pd.Timedelta(days=1)
    history_start = as_of_date - pd.Timedelta(days=history_days)
    prediction_start_date = pd.to_datetime(prediction_start_date)
    prediction_end_date = pd.to_datetime(prediction_end_date)
    
    print(f"History start date: {history_start:%Y-%m-%d}")
    print(f"Data as of date: {as_of_date:%Y-%m-%d}")
    print(f"Prediction period: {prediction_start_date:%Y-%m-%d} to {prediction_end_date:%Y-%m-%d}")

    # Transactions in prediction period
    prediction_period_mask = (transactions['t_dat'] >= prediction_start_date) & (transactions['t_dat'] <= prediction_end_date)
    transactions_prediction_period = transactions[prediction_period_mask]
    prediction_transactions_count = len(transactions_prediction_period)
    print(f"Transactions in prediction period: {len(transactions_prediction_period):,}")

    # Transactions before prediction period
    history_mask = (transactions['t_dat'] >= history_start) & (transactions['t_dat'] <= as_of_date)
    transactions_before_prediction = transactions[history_mask]
    print(f"Transaction rows before prediction: {len(transactions_before_prediction):,}")

    transactions_df = pd.concat([transactions_prediction_period, transactions_before_prediction])
    print(f"Transaction rows: {len(transactions_df):,}")
    
    # Pull all customers and articles that are in the sampled transactions
    sample_customer_ids = transactions_df['customer_id'].unique()
    sample_article_ids = transactions_df['article_id'].unique()
    
    customers_sample = customers[customers['customer_id'].isin(sample_customer_ids)]
    articles_sample = articles[articles['article_id'].isin(sample_article_ids)]
    
    customers_df = customers_sample
    articles_df = articles_sample

    
    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")

    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                            customer_features_df=customer_features,
                                            product_features_df=product_features,
                                            customer_fill_values=cfe.get_fill_values(),
                                            product_fill_values=pfe.get_fill_values()
                                           )
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=negative_ratio,
                                            random_state=random_state)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")

    unique_pairs = transactions_prediction_period[['customer_id', 'article_id']].drop_duplicates()
    missing_products = set(unique_pairs['article_id'].unique()) - set(product_features['article_id'].unique())
    missing_product_pairs = unique_pairs[unique_pairs['article_id'].isin(missing_products)].shape[0]
    missing_customers = set(unique_pairs['customer_id'].unique()) - set(customer_features['customer_id'].unique())
    print(f"Unique customer-product pairs in prediction period: {unique_pairs.shape[0]:,}")
    print(f"  - Duplicate pairs dropped: {prediction_transactions_count - unique_pairs.shape[0]:,}")
    print(f"  - Products filled with sentinels: {len(missing_products):,} products ({missing_product_pairs:,} pairs affected)")
    print(f"  - Customers filled with sentinels: {len(missing_customers):,} customers")
    print(f"  - Final positives: {(data['purchased'] == 1).sum():,}")

    return data

In [7]:
print("========= TRAINING DATA 1 =========")
train_prediction_start_date_1 = "2019-02-01"
train_prediction_end_date_1 = "2019-02-14"
train_data_1 =  build_product_recommendation_data(train_prediction_start_date_1, train_prediction_end_date_1,
                                                history_days=HISTORY_DAYS, negative_ratio=1)

========= TRAINING DATA 1 =========
History start date: 2019-01-01
Data as of date: 2019-01-31
Prediction period: 2019-02-01 to 2019-02-14
Transactions in prediction period: 33,292
Transaction rows before prediction: 80,485
Transaction rows: 113,777
Customer rows: 65,839
Product rows: 16,120
Total rows: 66,424
- Positives: 33,212
- Negatives: 33,212
Unique customer-product pairs in prediction period: 33,212
  - Duplicate pairs dropped: 80
  - Products filled with sentinels: 3,011 products (5,498 pairs affected)
  - Customers filled with sentinels: 6,023 customers
  - Final positives: 33,212


In [8]:
print("========= TRAINING DATA 2 =========")
train_prediction_start_date_2 = "2019-04-01"
train_prediction_end_date_2 = "2019-04-14"
train_data_2 =  build_product_recommendation_data(train_prediction_start_date_2, train_prediction_end_date_2,
                                                history_days=HISTORY_DAYS, negative_ratio=1)

========= TRAINING DATA 2 =========
History start date: 2019-03-01
Data as of date: 2019-03-31
Prediction period: 2019-04-01 to 2019-04-14
Transactions in prediction period: 41,810
Transaction rows before prediction: 81,127
Transaction rows: 122,937
Customer rows: 70,281
Product rows: 15,342
Total rows: 83,442
- Positives: 41,721
- Negatives: 41,721
Unique customer-product pairs in prediction period: 41,721
  - Duplicate pairs dropped: 89
  - Products filled with sentinels: 2,967 products (6,441 pairs affected)
  - Customers filled with sentinels: 7,362 customers
  - Final positives: 41,721


In [9]:
print("========= TRAINING DATA 3 =========")
train_prediction_start_date_3 = "2019-06-01"
train_prediction_end_date_3 = "2019-06-14"
train_data_3 =  build_product_recommendation_data(train_prediction_start_date_3, train_prediction_end_date_3,
                                                history_days=HISTORY_DAYS, negative_ratio=1)

========= TRAINING DATA 3 =========
History start date: 2019-05-01
Data as of date: 2019-05-31
Prediction period: 2019-06-01 to 2019-06-14
Transactions in prediction period: 44,632
Transaction rows before prediction: 98,958
Transaction rows: 143,590
Customer rows: 80,628
Product rows: 16,400
Total rows: 89,070
- Positives: 44,535
- Negatives: 44,535
Unique customer-product pairs in prediction period: 44,535
  - Duplicate pairs dropped: 97
  - Products filled with sentinels: 2,464 products (6,596 pairs affected)
  - Customers filled with sentinels: 7,908 customers
  - Final positives: 44,535


In [10]:
train_data = pd.concat([train_data_1, train_data_2, train_data_3], ignore_index=True)
print(f"Total rows: {len(train_data):,}")
print(f"- Positives: {(train_data['purchased'] == 1).sum():,}")
print(f"- Negatives: {(train_data['purchased'] == 0).sum():,}")

Total rows: 238,936
- Positives: 119,468
- Negatives: 119,468


In [11]:
print("========= VALIDATION DATA =========")
val_prediction_start = "2019-08-01"
val_prediction_end = "2019-08-14"
val_data =  build_product_recommendation_data(val_prediction_start, val_prediction_end,
                                                history_days=HISTORY_DAYS, negative_ratio=5)

========= VALIDATION DATA =========
History start date: 2019-07-01
Data as of date: 2019-07-31
Prediction period: 2019-08-01 to 2019-08-14
Transactions in prediction period: 38,493
Transaction rows before prediction: 114,640
Transaction rows: 153,133
Customer rows: 84,817
Product rows: 16,802
Total rows: 230,262
- Positives: 38,377
- Negatives: 191,885
Unique customer-product pairs in prediction period: 38,377
  - Duplicate pairs dropped: 116
  - Products filled with sentinels: 2,303 products (4,329 pairs affected)
  - Customers filled with sentinels: 6,727 customers
  - Final positives: 38,377


In [12]:
print("========= TEST DATA =========")
test_prediction_start = "2019-10-01"
test_prediction_end = "2019-10-14"
test_data =  build_product_recommendation_data(test_prediction_start, test_prediction_end,
                                                history_days=HISTORY_DAYS, negative_ratio=5)

========= TEST DATA =========
History start date: 2019-08-31
Data as of date: 2019-09-30
Prediction period: 2019-10-01 to 2019-10-14
Transactions in prediction period: 36,935
Transaction rows before prediction: 79,694
Transaction rows: 116,629
Customer rows: 67,255
Product rows: 14,001
Total rows: 221,238
- Positives: 36,873
- Negatives: 184,365
Unique customer-product pairs in prediction period: 36,873
  - Duplicate pairs dropped: 62
  - Products filled with sentinels: 2,806 products (4,936 pairs affected)
  - Customers filled with sentinels: 7,534 customers
  - Final positives: 36,873


In [13]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- age
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity
- is_new_customer
- FN
- Active
- club_member_status_NOT_ACTIVE_MEMBER
- club_member_status_PRE-CREATE
- fashion_news_frequency_REGULARLY


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [14]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 208
Unique garment groups: 21


In [15]:
train_data = train_data.drop('primary_department', axis=1, errors='ignore')
val_data = val_data.drop('primary_department', axis=1, errors='ignore')
test_data = test_data.drop('primary_department', axis=1, errors='ignore')

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment', dtype=int)
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment', dtype=int)
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment', dtype=int)

all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'garment_Jersey Basic', 'garment_Trousers', 'age', 'max_price', 'avg_days_between_purchases', 'garment_Under-, Nightwear', 'garment_Jersey Fancy', 'garment_Dresses Ladies', 'garment_Knitwear', 'garment_Shirts', 'garment_Dresses/Skirts girls', 'Active', 'garment_Accessories', 'avg_transaction_value', 'garment_Blouses', 'fashion_news_frequency_REGULARLY', 'garment_Shorts', 'garment_Dressed', 'min_price', 'days_since_last_purchase', 'days_since_last_sale', 'purchased', 'product_price_std', 'customer_id', 'FN', 'days_since_first_sale', 'garment_Skirts', 'garment_Shoes', 'category_diversity', 'total_spent', 'garment_Trousers Denim', 'club_member_status_NOT_ACTIVE_MEMBER', 'sales_last_30_days', 'customer_price_std', 'num_purchases', 'sales_last_7_days', 'garment_Socks and Tights', 'avg_price', 'garment_Special Offers', 'garment_Unknown', 'garment_Woven/Jersey/Knitted mix Baby', 'is_new_customer', 'garment_Swimwear', 'article_id', 'club_member_status_PRE-CREATE', 'garment_Outdoor'}


## Impute Missing values

There are too many missing values to use a mean or median. It overwhelms the distribution and turns the variable into noise.
Filling with -1 and making an indicator variable to flag it to the model.


Several features use `999` as a flag for customers or products with no purchase history:
- `days_since_last_purchase` — customers who have never purchased
- `avg_days_between_purchases` — customers with only one purchase (no gap to compute)
- `days_since_last_sale` — products that have never been sold
- `days_since_first_sale` — products that have never been sold

A mean or median value is not appropriate for these becuase some are so numerous that they would overwhelm the actual mean value and become meaningless. Or using positive value would imply events that did not happen such as a sale event.

**Strategy: indicator + constant**  
For each affected column we:
1. Add a binary indicator column (e.g., `days_since_last_purchase_missing = 1`) so the model can explicitly learn that the value is absent
2. Replace `999` with `-1` so the original column remains numeric and the sentinel doesn't inflate the feature's scale

In [16]:
print("Columns with indicator value")
indicator_cols = []
for col in train_data.columns:
    col_max = train_data[col].max()
    if col_max == 999:
        indicator_cols.append(col)
        print(f"- {col}")

# Remove the indicator values and replace with NaN
# Add an indicator col instead
print("Add indicator column and prepare to impute the fill value")
for col in indicator_cols:
    for df in [train_data, val_data, test_data]:
        df[f'{col}_missing'] = (df[col] == 999).astype(int)
        df[col] = df[col].replace(999, np.nan)
        
print("Impute missing values with constant -1")
imputer = SimpleImputer(strategy='constant', fill_value=-1)
train_data[indicator_cols] = imputer.fit_transform(train_data[indicator_cols])
val_data[indicator_cols] = imputer.transform(val_data[indicator_cols])
test_data[indicator_cols] = imputer.transform(test_data[indicator_cols])

Columns with indicator value
- avg_days_between_purchases
- days_since_last_purchase
- days_since_last_sale
- days_since_first_sale
Add indicator column and prepare to impute the fill value
Impute missing values with constant -1


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [17]:
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(238936, 47)
y_train.shape=(238936,)
X_val.shape=(230262, 47)
y_val.shape=(230262,)
X_test.shape=(221238, 47)
y_test.shape=(221238,)


## Save Processed Data
### as pickle files

In [18]:
# Save data as pickle to avoid reprocessing
processed_data_path = processed_path / 'product_recommendation'
processed_data_path.mkdir(parents=True, exist_ok=True)

with open(processed_data_path / 'X_train_base.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(processed_data_path / 'y_train_base.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(processed_data_path / 'X_val_base.pkl', 'wb') as f:
    pickle.dump(X_val, f)
with open(processed_data_path / 'y_val_base.pkl', 'wb') as f:
    pickle.dump(y_val, f)
with open(processed_data_path / 'X_test_base.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(processed_data_path / 'y_test_base.pkl', 'wb') as f:
    pickle.dump(y_test, f)